# Analyse des résultats de la recherche heuristique étendue (Étape 2)

Ce notebook trace les courbes de recherche à partir des résultats **réels** produits par `run_search.py` dans `MKAN/Etude_benchmarck/extended_search/` — aucune donnée fabriquée : si un fichier manque, la cellule correspondante l'indique explicitement plutôt que d'inventer un résultat.

Sources utilisées :
- `search_history.csv` — une ligne par évaluation (généré par `results_export.py`), déjà normalisé (colonnes plates par porte/base).
- `heuristic_best_config.json` — représentant retenu du front de Pareto (section 9 du mémoire).
- `top5_configurations.json` — 5 meilleures configurations distinctes.
- `pareto_front.json` — front non-dominé complet (NSGA-II, si présent dans ce run).

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RESULTS_DIR = "."  # ce notebook vit dans MKAN/Etude_benchmarck/ ; les résultats sont dans extended_search/ à côté
RESULTS_DIR = os.path.join(RESULTS_DIR, "extended_search")

def _load_json(name):
    path = os.path.join(RESULTS_DIR, name)
    if not os.path.exists(path):
        print(f"[absent] {name} — cellule(s) correspondante(s) sautées, rien n'est inventé.")
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

history_path = os.path.join(RESULTS_DIR, "search_history.csv")
assert os.path.exists(history_path), f"search_history.csv introuvable dans {RESULTS_DIR}"
df = pd.read_csv(history_path)

best_config = _load_json("heuristic_best_config.json")
top5 = _load_json("top5_configurations.json") or _load_json("top5_configs.json")
pareto_front = _load_json("pareto_front.json")

FIGURES_DIR = os.path.join(RESULTS_DIR, "notebook_figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(fig, name):
    """Sauvegarde une figure en .html (interactif, toujours possible) ET en
    .png (statique, via kaleido — déjà présent dans le venv, utilisable
    directement dans le mémoire LaTeX). Avant ce correctif, fig.show() ne
    faisait qu'afficher en ligne dans la sortie de la cellule : rien n'était
    écrit sur disque."""
    html_path = os.path.join(FIGURES_DIR, f"{name}.html")
    png_path = os.path.join(FIGURES_DIR, f"{name}.png")
    fig.write_html(html_path)
    try:
        fig.write_image(png_path, scale=2)
    except Exception as exc:   # kaleido peut manquer selon l'environnement — HTML reste disponible
        print(f"  [avertissement] export PNG impossible pour {name} ({exc}) — HTML seul disponible.")
        png_path = None
    print(f"Sauvegardé : {html_path}" + (f" et {png_path}" if png_path else ""))

print(f"{len(df)} évaluations chargées, {df['generation'].nunique()} générations, "
      f"{df['Cell_Type'].nunique()} type(s) de cellule, {df['Input_Regime'].nunique()} régime(s) d'entrée.")
print(f"Figures sauvegardées dans : {FIGURES_DIR}")
df.head()

## 1. Convergence globale

Meilleure fitness et fitness moyenne par génération, bande ±1 écart-type — mêmes conventions que `RechercheHeuristique.plot_convergence()` (heuristic_search.py).

In [ ]:
stats = df.groupby("generation")["fitness"].agg(["max", "mean", "std"]).reset_index()
stats["std"] = stats["std"].fillna(0.0)

x = stats["generation"].tolist()
y_mean = stats["mean"].tolist()
y_std = stats["std"].tolist()
y_upper = [m + s for m, s in zip(y_mean, y_std)]
y_lower = [m - s for m, s in zip(y_mean, y_std)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x + x[::-1], y=y_upper + y_lower[::-1], fill="toself",
                          fillcolor="rgba(255,152,0,0.15)", line=dict(color="rgba(0,0,0,0)"),
                          name="±1σ", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=x, y=y_mean, mode="lines", name="Fitness moyenne",
                          line=dict(color="#FF9800", width=2, dash="dot")))
fig.add_trace(go.Scatter(x=x, y=stats["max"].tolist(), mode="lines+markers",
                          name="Meilleure de la génération", line=dict(color="#2196F3", width=2.5)))
best_overall = df["fitness"].max()
fig.add_hline(y=best_overall, line_dash="dash", line_color="#4CAF50", line_width=1.5,
              annotation_text=f"Best global = {best_overall:.4f}", annotation_position="bottom right")
fig.update_layout(title="Convergence de la recherche heuristique étendue (données réelles)",
                   xaxis_title="Génération", yaxis_title="Fitness (AHP, §3.3.2)",
                   template="plotly_white", legend=dict(orientation="h", y=-0.2))
fig.show()
save_fig(fig, "01_convergence_globale")

## 2. Convergence par architecture — TKANCell vs GRUKANCell

Compare les deux cellules à budget paramétrique équivalent (`hidden_size_gru = floor((4/3)·hidden_size)`, section 3.4 du mémoire).

In [ ]:
def plot_best_by_group(df, group_col, colors=None, title=""):
    colors = colors or {}
    palette = ["#2196F3", "#F44336", "#4CAF50", "#9C27B0", "#FF9800"]
    fig = go.Figure()
    for i, (group_val, sub) in enumerate(df.groupby(group_col)):
        g_stats = sub.groupby("generation")["fitness"].max().reset_index()
        # courbe monotone du meilleur cumulé — permet de comparer la vitesse de convergence
        g_stats["cummax"] = g_stats["fitness"].cummax()
        color = colors.get(group_val, palette[i % len(palette)])
        fig.add_trace(go.Scatter(x=g_stats["generation"], y=g_stats["cummax"],
                                  mode="lines+markers", name=f"{group_val} (n={len(sub)})",
                                  line=dict(color=color, width=2.5)))
    fig.update_layout(title=title, xaxis_title="Génération",
                       yaxis_title="Meilleure fitness cumulée", template="plotly_white",
                       legend=dict(orientation="h", y=-0.2))
    return fig

fig = plot_best_by_group(df, "Cell_Type",
                          colors={"TKANCell": "#2196F3", "GRUKANCell": "#F44336"},
                          title="Convergence comparée — TKANCell vs GRUKANCell")
fig.show()
save_fig(fig, "02_convergence_cell_type")

## 3. Convergence par régime d'entrée — brut vs traité

Les deux régimes sont évalués séparément, jamais concaténés (section "Régime d'entrée comme facteur transversal", step_2).

In [ ]:
fig = plot_best_by_group(df, "Input_Regime",
                          colors={"raw": "#FF9800", "engineered": "#4CAF50"},
                          title="Convergence comparée — régime brut vs traité")
fig.show()
save_fig(fig, "03_convergence_input_regime")

## 4. Distribution de la fitness par base KAN, par porte

Un panneau par porte (Forget/Input/Candidate/Output) — montre quelle base KAN tend à produire les meilleures configurations pour CHAQUE porte, conformément à la cartographie théorique de l'Étape 1 (seuillage quasi-binaire / signal haute fréquence / projection quasi-linéaire).

In [ ]:
gates = ["Base_Forget", "Base_Input", "Base_Candidate", "Base_Output"]
gates = [g for g in gates if g in df.columns]

fig = make_subplots(rows=1, cols=len(gates), subplot_titles=[g.replace("Base_", "") for g in gates])
for col_idx, gate_col in enumerate(gates, start=1):
    sub = df.dropna(subset=[gate_col])   # NaN = porte absente pour cet individu (ex. Output en GRUKANCell)
    order = sub.groupby(gate_col)["fitness"].median().sort_values(ascending=False).index
    for base in order:
        vals = sub.loc[sub[gate_col] == base, "fitness"]
        fig.add_trace(go.Box(y=vals, name=base, showlegend=False,
                              marker_color="#2196F3", line_color="#1565C0"),
                      row=1, col=col_idx)
fig.update_layout(title="Distribution de la fitness par base KAN, par porte (données réelles)",
                   template="plotly_white", height=450)
fig.show()
save_fig(fig, "04_distribution_base_par_porte")

## 5. Compromis multi-critères (MCC vs latence, PR-AUC vs Brier)

Vue brute des objectifs (avant agrégation AHP) — permet de voir si la fitness scalaire cache des compromis. Coloré par type de cellule.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["MCC_raw vs latence", "PR-AUC vs Brier"])
color_map = {"TKANCell": "#2196F3", "GRUKANCell": "#F44336"}
for cell_type, sub in df.groupby("Cell_Type"):
    c = color_map.get(cell_type, "#9C27B0")
    fig.add_trace(go.Scatter(x=sub["latency_ms"], y=sub["MCC_raw"], mode="markers",
                              name=cell_type, marker=dict(color=c, size=7, opacity=0.7),
                              legendgroup=cell_type),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=sub["Brier"], y=sub["PR_AUC"], mode="markers",
                              name=cell_type, marker=dict(color=c, size=7, opacity=0.7),
                              legendgroup=cell_type, showlegend=False),
                  row=1, col=2)
fig.add_vline(x=100, line_dash="dash", line_color="black", row=1, col=1,
              annotation_text="seuil 100 ms")
fig.update_xaxes(title_text="Latence (ms)", row=1, col=1)
fig.update_yaxes(title_text="MCC_raw", row=1, col=1)
fig.update_xaxes(title_text="Brier (plus bas = mieux)", row=1, col=2)
fig.update_yaxes(title_text="PR-AUC", row=1, col=2)
fig.update_layout(title="Compromis entre objectifs bruts (données réelles)",
                   template="plotly_white", height=450)
fig.show()
save_fig(fig, "05_compromis_objectifs")

## 6. Front de Pareto (NSGA-II)

Nécessite `pareto_front.json` (produit par `ExtendedHeuristicSearch.export_pareto_front()`). S'il est absent de ce run, la cellule l'indique — rien n'est reconstruit à partir d'une approximation.

In [ ]:
if pareto_front is None:
    print("pareto_front.json absent de ce run — cette cellule ne peut rien tracer sans lui "
          "(ne pas reconstruire un front approximatif à partir de search_history.csv, ce ne serait "
          "pas rigoureusement le même calcul que le tri par dominance de la recherche).")
else:
    front = pareto_front["front"]
    fc = [c["fitness_components"] for c in front]
    labels = [c["individual"].get("Cell_Type", "?") for c in front]
    fig = go.Figure(go.Scatter(
        x=[f["MCC_clipped"] for f in fc], y=[f["R2_symbolic"] for f in fc],
        mode="markers+text", text=labels, textposition="top center",
        marker=dict(size=10, color=[f["Brier"] for f in fc], colorscale="Viridis_r",
                    showscale=True, colorbar=dict(title="Brier"))))
    fig.update_layout(title=f"Front de Pareto ({pareto_front['metadata']['n_front']} points) — "
                             "MCC_clipped vs R²-symbolic, couleur = Brier",
                       xaxis_title="MCC_clipped (max)", yaxis_title="R²_symbolic (max)",
                       template="plotly_white")
    fig.show()
    save_fig(fig, "06_pareto_front")

## 7. Top-5 configurations distinctes

In [8]:
if top5 is None:
    print("top5_configurations.json / top5_configs.json absent de ce run.")
else:
    rows = []
    for i, entry in enumerate(top5, start=1):
        ind = entry.get("individual", entry)
        rows.append({
            "rang": i, "score": entry.get("score", entry.get("fitness_total")),
            "Cell_Type": ind.get("Cell_Type"), "Input_Regime": ind.get("Input_Regime"),
            "hidden_size": ind.get("hidden_size"),
            "Base_Forget": ind.get("gates", {}).get("Forget", {}).get("Base"),
            "Base_Input": ind.get("gates", {}).get("Input", {}).get("Base"),
            "Base_Candidate": ind.get("gates", {}).get("Candidate", {}).get("Base"),
            "Base_Output": ind.get("gates", {}).get("Output", {}).get("Base"),
        })
    display(pd.DataFrame(rows))

,rang,score,Cell_Type,Input_Regime,hidden_size,Base_Forget,Base_Input,Base_Candidate,Base_Output
0,1,0.694990,GRUKANCell,raw,128,relukan,efficientkan,wavkan,None
1,2,0.687820,TKANCell,raw,32,relukan,chebyshev,wavkan,linear
2,3,0.676536,GRUKANCell,engineered,16,chebyshev,hybrid,fourier,None
3,4,0.671550,TKANCell,raw,32,efficientkan,fasterkan,wavkan,efficientkan
4,5,0.670895,TKANCell,raw,32,efficientkan,efficientkan,wavkan,linear


## 8. Configuration retenue (représentant du front, `heuristic_best_config.json`)

In [9]:
if best_config is None:
    print("heuristic_best_config.json absent de ce run.")
else:
    print(json.dumps(best_config["theta_opt"], indent=2, ensure_ascii=False))
    print(json.dumps(best_config["theta_struct"], indent=2, ensure_ascii=False))
    print(json.dumps(best_config["fitness_components"], indent=2, ensure_ascii=False))

{
  "hidden_size": 128,
  "lr": 0.0001,
  "lam": 0.0001,
  "mu1": 0.1,
  "mu2": 0.6989017708691363,
  "batch_size": 32,
  "W": 5
}
{
  "cell_type": "GRUKANCell",
  "hidden_size_gru_adjusted": 170,
  "input_regime": "raw",
  "gates": {
    "Forget": {
      "base": "relukan",
      "hyperparams": {
        "Grid_G": 8
      }
    },
    "Input": {
      "base": "efficientkan",
      "hyperparams": {
        "Grid_G": 3,
        "Spline_Degree_k": 4
      }
    },
    "Candidate": {
      "base": "wavkan",
      "hyperparams": {
        "Mother_Wavelet": "dog",
        "M_wavelets": 3
      }
    }
  }
}
{
  "MCC_raw": 0.9834131726178621,
  "MCC_clipped": 0.9834131726178621,
  "PR_AUC": 0.9993591039197686,
  "Brier": 0.004768696238500933,
  "R2_symbolic": 0.525,
  "latency_ms": 853.5887077450752,
  "penalty_lat": 567.8959404408924,
  "fitness_total": 0.6949897405913118
}
